# 🥔 Potato Leaf Disease Detection (CNN)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shantanuparekh/potato-leaf-disease-detection/blob/main/potato_disease_colab.ipynb)

Classifies a potato leaf photo into one of three classes:
**Early Blight**, **Late Blight** or **Healthy**.

**Two ways to use this notebook**
- **Option A – Quick demo (no training):** load the pretrained model from the GitHub repo and test your own images.
- **Option B – Train it yourself:** train the CNN on your own copy of the dataset (80% train / 20% test).

> Tip: *Runtime → Change runtime type → T4 GPU* makes training much faster.

## 1. Setup

In [ ]:
import os, json, shutil, subprocess
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

IMG_SIZE = (128, 128)
CLASS_NAMES = ["Potato___Early_blight", "Potato___Late_blight", "Potato___healthy"]

## 2A. Option A – Load the pretrained model (no training needed)
Edit `GITHUB_USER` to the account that hosts this repo. Skip this section if you want to train (Option B).

In [ ]:
GITHUB_USER = "shantanuparekh"          # <-- change this
REPO = "potato-leaf-disease-detection"

if not os.path.exists(REPO):
    subprocess.run(["git", "clone", f"https://github.com/shantanuparekh/potato-leaf-disease-detection.git"], check=True)

model = tf.keras.models.load_model(f"{REPO}/models/potato_model.keras")
with open(f"{REPO}/models/class_names.json") as f:
    class_names = json.load(f)
print("Model loaded. Classes:", class_names)

## 2B. Option B – Train your own model
### Step 1: Get the dataset into Google Drive
Put the dataset folder in your Drive, for example:
`MyDrive/Project/Plant_leave_diseases_dataset_with_augmentation/`
It must contain the three class folders: `Potato___Early_blight`, `Potato___Late_blight`, `Potato___healthy`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ---- EDIT THESE PATHS ----
DRIVE_DATA_DIR = "/content/drive/MyDrive/Project/Plant_leave_diseases_dataset_with_augmentation"
OUTPUT_DIR     = "/content/drive/MyDrive/Project_output"     # model + graphs are saved here

BATCH_SIZE = 32
EPOCHS = 20
SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Copy only the 3 potato folders to Colab's fast local disk (Drive is slow for many small files)
DATA_DIR = "/content/dataset"
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)
    for c in CLASS_NAMES:
        shutil.copytree(os.path.join(DRIVE_DATA_DIR, c), os.path.join(DATA_DIR, c))
for c in CLASS_NAMES:
    print(c, "->", len(os.listdir(os.path.join(DATA_DIR, c))), "images")

### Step 2: Load data (80% train / 20% test)

In [ ]:
train_full = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.20, subset="training", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="int",
    class_names=CLASS_NAMES)

# the 20% TEST set (same seed => no overlap with training data)
test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.20, subset="validation", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="int",
    class_names=CLASS_NAMES)

class_names = train_full.class_names
num_classes = len(class_names)

# 10% of the training part is used as validation while training
n_batches = tf.data.experimental.cardinality(train_full).numpy()
val_batches = max(1, int(0.10 * n_batches))
val_ds = train_full.take(val_batches)
train_ds = train_full.skip(val_batches)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

### Step 3: Build the CNN

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
])

model = models.Sequential([
    layers.Input(shape=IMG_SIZE + (3,)),
    data_augmentation,
    layers.Rescaling(1.0 / 255),

    layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu"), layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax"),
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

### Step 4: Train

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(os.path.join(OUTPUT_DIR, "best_model.keras"),
                                       monitor="val_accuracy", save_best_only=True),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

### Step 5: Evaluate on the 20% test set

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Accuracy: {test_acc * 100:.2f}%")

y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

print(classification_report(y_true, y_pred, target_names=class_names))

fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ax[0].plot(history.history["accuracy"], label="Train")
ax[0].plot(history.history["val_accuracy"], label="Validation")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(history.history["loss"], label="Train")
ax[1].plot(history.history["val_loss"], label="Validation")
ax[1].set_title("Loss"); ax[1].set_xlabel("Epoch"); ax[1].legend()
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt="d", cmap="Greens",
            xticklabels=class_names, yticklabels=class_names, ax=ax[2])
ax[2].set_title("Confusion Matrix (test set)"); ax[2].set_xlabel("Predicted"); ax[2].set_ylabel("Actual")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "results.png"))
plt.show()

### Step 6: Save the model

In [ ]:
model.save(os.path.join(OUTPUT_DIR, "potato_model.keras"))
with open(os.path.join(OUTPUT_DIR, "class_names.json"), "w") as f:
    json.dump(class_names, f)
print("Saved to", OUTPUT_DIR)

# Optional: download straight to your computer
# from google.colab import files
# files.download(os.path.join(OUTPUT_DIR, "potato_model.keras"))

## 3. Test it on your own images
Run this after **either** Option A or Option B. Upload one or more leaf photos (jpg/png).

In [ ]:
from google.colab import files
from PIL import Image

INFO = {
    "Potato___Early_blight": "Early Blight - remove infected leaves, use a recommended fungicide, rotate crops.",
    "Potato___Late_blight":  "Late Blight - spreads fast! Remove infected plants and apply fungicide immediately.",
    "Potato___healthy":      "Healthy leaf - no disease detected.",
}

def predict_image(path):
    img = Image.open(path).convert("RGB")
    arr = np.expand_dims(np.array(img.resize(IMG_SIZE), dtype="float32"), 0)   # model rescales internally
    probs = model.predict(arr, verbose=0)[0]
    idx = int(np.argmax(probs))
    return img, class_names[idx], float(probs[idx]) * 100, probs

uploaded = files.upload()
for name in uploaded:
    img, cls, conf, probs = predict_image(name)
    plt.figure(figsize=(4, 4)); plt.imshow(img); plt.axis("off")
    plt.title(f"{cls.replace('Potato___', '')}  ({conf:.1f}%)")
    plt.show()
    for c, p in zip(class_names, probs):
        print(f"  {c:<24} {p * 100:6.2f}%")
    print(" ", INFO.get(cls, ""))
    if conf < 70:
        print("  Low confidence - the image may be unclear or not a potato leaf.")